In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

# ------------------------
# Configuration MinIO
# ------------------------
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# ------------------------
# Initialisation Spark
# ------------------------
spark = (SparkSession.builder
         .appName("TelcoChurn_Gold")
         .master("spark://spark-master:7077")
         .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
         .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
         .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}")
         .config("spark.hadoop.fs.s3a.path.style.access", "true")
         .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
         .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
         .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
         .config("spark.driver.memory", "3g")
         .config("spark.executor.memory", "2g")
         .config("spark.executor.cores", "4")
         .config("spark.executor.instances", "2")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ------------------------
# Charger Silver Layer
# ------------------------
silver_path = f"s3a://{MINIO_BUCKET}/silver/telco_churn"
df = spark.read.format("delta").load(silver_path)
print(f"✅ Silver Layer chargé : {df.count()} lignes")

# ------------------------
# Colonnes catégorielles
# ------------------------
categorical_cols = ["Contract", "PaymentMethod", "InternetService"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

# ------------------------
# Colonnes numériques
# ------------------------
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df = df.select("*", *[col(c).cast("double").alias(c) for c in num_cols])

assembler_num = VectorAssembler(inputCols=num_cols, outputCol="num_unscaled")
scaler = StandardScaler(inputCol="num_unscaled", outputCol="num_scaled", withMean=False, withStd=True)

# ------------------------
# Pipeline complet
# ------------------------
pipeline = Pipeline(stages=indexers + encoders + [assembler_num, scaler])
model = pipeline.fit(df)
df_gold_tmp = model.transform(df)

# ------------------------
# Assemblage final des features
# ------------------------
all_features = [f"{c}_ohe" for c in categorical_cols] + ["num_scaled"] + [
    "HasInternetService", "HasPhoneService", "MultipleLines",
    "Partner", "Dependents", "PaperlessBilling"
]

final_assembler = VectorAssembler(inputCols=all_features, outputCol="features")
df_gold = final_assembler.transform(df_gold_tmp)

# ------------------------
# Sauvegarde Gold Layer
# ------------------------
gold_path = f"s3a://{MINIO_BUCKET}/gold/telco_churn"
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)
print("✅ Gold Layer enregistré :", gold_path)

# Affichage des 5 premières lignes
df_gold.show(5, truncate=False)


25/11/25 15:51:48 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/11/25 15:51:48 WARN DeltaLog: Failed to parse s3a://telco-churn/silver/telco_churn/_delta_log/_last_checkpoint. This may happen if there was an error during read operation, or a file appears to be partial. Sleeping and trying again.
java.lang.NullPointerException: Cannot invoke "org.apache.spark.SparkEnv.conf()" because the return value of "org.apache.spark.SparkEnv$.get()" is null
	at org.apache.spark.sql.delta.storage.DelegatingLogStore.<init>(DelegatingLogStore.scala:38)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStoreWithClassName(LogStore.scala:288)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore(LogStore.scala:385)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore$(LogStore.scala:380)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStore(LogStore.scala:266)
	at org.apache.spark.sql.delta.sto

Py4JJavaError: An error occurred while calling o69.load.
: java.lang.NullPointerException: Cannot invoke "org.apache.spark.SparkEnv.conf()" because the return value of "org.apache.spark.SparkEnv$.get()" is null
	at org.apache.spark.sql.delta.storage.DelegatingLogStore.<init>(DelegatingLogStore.scala:38)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStoreWithClassName(LogStore.scala:288)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore(LogStore.scala:385)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore$(LogStore.scala:380)
	at org.apache.spark.sql.delta.storage.LogStore$.createLogStore(LogStore.scala:266)
	at org.apache.spark.sql.delta.storage.LogStore$.apply(LogStore.scala:279)
	at org.apache.spark.sql.delta.storage.LogStore$.apply(LogStore.scala:274)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore(LogStore.scala:322)
	at org.apache.spark.sql.delta.storage.LogStoreProvider.createLogStore$(LogStore.scala:321